# 🤖 Build your AI Assistant with LLMs & RAG
## Workshop GoMyCode — Guide pédagogique complet

---

### 📋 Table des matières
1. [Connaissances Fondamentales](#partie-1)
2. [Chatbot IA avec un LLM](#partie-2)
3. [Chatbot IA avec LLMs & RAG](#partie-3)

---

> **⏱️ Durée estimée :** 4 heures  
> **🎯 Objectif :** Construire un assistant IA capable de répondre à des questions basées sur vos propres documents

---
## ⚙️ Installation des dépendances

Avant de commencer, installez toutes les bibliothèques nécessaires.

In [ ]:
# Installation de toutes les bibliothèques nécessaires
!pip install -q langchain langchain-community langchain-nvidia-ai-endpoints \
    faiss-cpu tiktoken pypdf sentence-transformers \
    python-dotenv gradio

---
<a id='partie-1'></a>
# 📚 Partie 1 — Connaissances Fondamentales

Avant de coder, il est important de comprendre les concepts clés.

## 1.1 Qu'est-ce que l'Intelligence Artificielle (IA) ?

> **Définition :** L'IA consiste à rendre les ordinateurs et les machines **intelligents**. C'est comme apprendre à un ordinateur à **penser**, **apprendre** et **résoudre des problèmes**, comme un être humain.

### Exemples du quotidien :
| Exemple | Description |
|---------|-------------|
| 💬 Chatbots | Répondent aux questions des clients (ex: support client) |
| 🚗 Voitures autonomes | Tesla, Waymo — conduisent sans conducteur humain |
| 📱 Reconnaissance faciale | Déverrouille votre smartphone |
| 🎵 Recommandations | Netflix, Spotify suggèrent des contenus personnalisés |

## 1.2 Qu'est-ce que l'IA Générative ?

> **Définition :** L'IA Générative est un type d'IA qui peut **créer du nouveau contenu** (texte, images, musique, vidéos), plutôt que simplement analyser ou classifier des données existantes.

```
IA Classique              IA Générative
───────────────────       ──────────────────────────
Entrée → Analyse          Entrée → CRÉATION de nouveau contenu
"Est-ce un chat ?"        "Décris un chat imaginaire"
Réponse: OUI/NON          Réponse: Un texte descriptif créatif
```

### Exemples d'IA Générative :
- **ChatGPT** — génère du texte
- **DALL-E / Midjourney** — génère des images
- **Suno / Udio** — génère de la musique
- **Sora** — génère des vidéos

## 1.3 Qu'est-ce qu'un LLM (Large Language Model) ?

> **Définition :** Un LLM est un modèle entraîné principalement sur du **texte** pour **comprendre** et **générer** du langage naturel.

```
┌─────────────────────────────────────┐
│             LLM                     │
│   Entrée: 📝 Texte → Sortie: 📝 Texte│
└─────────────────────────────────────┘
```

### LLMs populaires :
| Modèle | Créateur | Points forts |
|--------|----------|--------------|
| GPT-4o | OpenAI | Polyvalent, très puissant |
| Claude 3 | Anthropic | Long contexte, sûr |
| Gemini | Google | Multimodal |
| **Llama 3.1** ⭐ | **Meta / NVIDIA** | **Open source — utilisé dans ce workshop** |
| Mistral | Mistral AI | Léger, open source |

## 1.4 Qu'est-ce qu'un Prompt ?

> **Définition :** Un prompt est un **court texte** que vous donnez à l'IA pour lui indiquer ce que vous voulez qu'elle crée ou fasse.

### Règle d'or : **Plus le prompt est précis, meilleur est le résultat !**

| Type de prompt | Exemple |
|---------------|--------|
| ❌ Basique | "Écris un email à un client." |
| ✅ Bon prompt | "Écris un email professionnel à un client nommé Sarah pour lui expliquer que le rapport du projet sera livré vendredi au lieu de mercredi, en présentant des excuses et en proposant un suivi." |

In [ ]:
# 🧪 Exercice 1.4 — Comparaison de prompts
# Observez la différence entre un prompt basique et un prompt précis

prompt_basique = "Écris un email à un client."

prompt_avance = """Écris un email professionnel à un client nommé Sarah.
Contexte : Le rapport du projet sera livré vendredi au lieu de mercredi.
Ton : Professionnel et empathique.
Inclure : Des excuses sincères et une proposition de suivi téléphonique."""

print("📝 PROMPT BASIQUE:")
print("-" * 40)
print(prompt_basique)
print()
print("✅ PROMPT AVANCÉ:")
print("-" * 40)
print(prompt_avance)

---
<a id='partie-2'></a>
# 🤖 Partie 2 — Chatbot IA avec un LLM

Construisons notre premier chatbot en utilisant directement un LLM.

## 2.1 Configuration de l'API

Pour interagir avec un LLM, nous avons besoin d'une **clé API**.

> 💡 **Conseil sécurité :** Ne jamais écrire votre clé API directement dans le code ! Utilisez des variables d'environnement.

In [ ]:
import os
from getpass import getpass

# Entrez votre clé API NVIDIA de manière sécurisée
# Obtenez votre clé gratuitement sur: https://build.nvidia.com
os.environ["NVIDIA_API_KEY"] = getpass("🔑 Entrez votre clé API NVIDIA: ")

print("✅ Clé API NVIDIA configurée avec succès!")

## 2.2 Premier appel à un LLM

Utilisons **LangChain** pour interagir avec GPT.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, SystemMessage

# Initialisation du modèle LLM — Llama 3.1 via NVIDIA API
llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",  # Modèle Llama 3.1 70B hébergé par NVIDIA
    temperature=0.7,                        # Créativité (0=déterministe, 1=très créatif)
    max_tokens=500                           # Longueur maximale de la réponse
)

print("✅ LLM initialisé:", llm.model)

In [ ]:
# Premier test : poser une question simple au LLM
messages = [
    SystemMessage(content="Tu es un assistant IA utile et pédagogique."),
    HumanMessage(content="Explique-moi ce qu'est un LLM en 3 phrases simples.")
]

reponse = llm.invoke(messages)

print("🤖 Réponse du LLM:")
print("-" * 50)
print(reponse.content)

## 2.3 Construire un Chatbot avec mémoire de conversation

Un vrai chatbot doit se **souvenir** du contexte de la conversation.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

class SimpleChatbot:
    """
    Un chatbot simple avec mémoire de conversation.
    """
    
    def __init__(self, system_prompt: str = "Tu es un assistant IA utile."):
        self.llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct", temperature=0.7)
        # Historique des messages (mémoire de la conversation)
        self.history = [
            SystemMessage(content=system_prompt)
        ]
    
    def chat(self, user_input: str) -> str:
        """Envoie un message et retourne la réponse."""
        # 1. Ajouter le message de l'utilisateur à l'historique
        self.history.append(HumanMessage(content=user_input))
        
        # 2. Envoyer tout l'historique au LLM
        response = self.llm.invoke(self.history)
        
        # 3. Sauvegarder la réponse dans l'historique
        self.history.append(AIMessage(content=response.content))
        
        return response.content
    
    def reset(self):
        """Réinitialise la conversation."""
        self.history = [self.history[0]]  # Garde uniquement le system prompt
        print("🔄 Conversation réinitialisée.")

# Créer notre chatbot
bot = SimpleChatbot(
    system_prompt="Tu es un assistant IA expert en technologie. Réponds de façon claire et concise en français."
)

print("✅ Chatbot créé avec succès!")

In [ ]:
# Test de la conversation avec mémoire
print("👤 Utilisateur: Bonjour! Qu'est-ce que l'IA générative?")
reponse1 = bot.chat("Bonjour! Qu'est-ce que l'IA générative?")
print(f"🤖 Bot: {reponse1}")
print()

print("👤 Utilisateur: Donne-moi 3 exemples concrets.")
reponse2 = bot.chat("Donne-moi 3 exemples concrets.")
print(f"🤖 Bot: {reponse2}")
print()

# Le bot se souvient du contexte précédent !
print("👤 Utilisateur: Lequel de ces exemples est le plus populaire?")
reponse3 = bot.chat("Lequel de ces exemples est le plus populaire?")
print(f"🤖 Bot: {reponse3}")

## 2.4 La Faiblesse des LLMs : Les Hallucinations ⚠️

> **Définition :** Une **hallucination** est quand l'IA génère des informations **incorrectes ou inventées** avec l'air d'être confiant.

### Pourquoi les LLMs hallucinent-ils ?
- Ils sont entraînés jusqu'à une certaine **date limite** (cutoff date)
- Ils n'ont **pas accès à Internet** en temps réel
- Ils génèrent des réponses **probabilistes**, pas factuelles
- Leurs données d'entraînement peuvent contenir des **erreurs**

In [ ]:
# Démonstration d'une hallucination potentielle
bot.reset()

question_piegee = "Quelle est la population exacte de Tunis en temps réel aujourd'hui?"

print(f"👤 Question: {question_piegee}")
print()
reponse = bot.chat(question_piegee)
print(f"🤖 Bot: {reponse}")
print()
print("⚠️  Problème: Le LLM peut donner un chiffre précis, mais il ne connaît pas")
print("   les données en temps réel. C'est une HALLUCINATION potentielle!")
print("\n💡 Solution: Le RAG ! (Partie 3)")

---
<a id='partie-3'></a>
# 🔍 Partie 3 — Chatbot IA avec LLMs & RAG

## Qu'est-ce que le RAG (Retrieval Augmented Generation) ?

> Le RAG est une technique qui améliore les LLMs en leur donnant accès à une **source de connaissance externe**.

```
                    ┌─────────────────────────────────────┐
                    │           PIPELINE RAG              │
                    └─────────────────────────────────────┘

  📄 Documents    ──►  ✂️ Chunking  ──►  🔢 Embeddings  ──►  🗄️ Vector DB
                                                                    │
  👤 Question  ──►  🔢 Embedding  ──────────────────────►  🔍 Recherche
                                                                    │
                                                                    ▼
                                           📝 Contexte pertinent trouvé
                                                                    │
                                                                    ▼
  👤 Question + 📝 Contexte  ──►  🤖 LLM  ──►  ✅ Réponse précise
```

## 3.1 Étape 1 — Chargement des Documents

La première étape est de charger vos documents sources.

In [ ]:
# Créons d'abord un document exemple sur lequel notre RAG va travailler
# Dans un cas réel, vous chargeriez des fichiers PDF, Word, etc.

from langchain.schema import Document

# Document exemple : Informations sur GoMyCode et le workshop
texte_exemple = """
GoMyCode est une startup éducative innovante qui forme les meilleurs talents 
aux dernières compétences numériques grâce à un modèle d'apprentissage hybride 
combinant une orientation en personne et une plateforme d'apprentissage en ligne.

GoMyCode est présent dans 12 villes en Tunisie: En ligne, El Mourouj, Bardo, 
Boumhel, Lac 1, El Menzah, Nabeul, Sousse, Kairouan, Gabes, Tozeur et Tataouine.

Le workshop "Build your AI Assistant with LLMs & RAG" est un workshop de 4 heures 
qui couvre les sujets suivants:
1. Connaissances Fondamentales sur l'IA
2. Chatbot IA avec un LLM
3. Chatbot IA avec LLMs & RAG

L'intelligence artificielle (IA) consiste à rendre les ordinateurs intelligents, 
capables de penser, d'apprendre et de résoudre des problèmes comme un être humain.

Un LLM (Large Language Model) est un modèle entraîné sur du texte pour comprendre 
et générer du langage naturel. Il prend du texte en entrée et produit du texte en sortie.

Le RAG (Retrieval Augmented Generation) est une technique qui améliore les LLMs 
en leur donnant accès à une source de connaissance externe. Contrairement aux modèles 
de langage traditionnels qui se basent uniquement sur leur entraînement, un système RAG 
récupère d'abord des informations pertinentes avant de produire une réponse.

Le chunking consiste à découper un document en petits segments appelés "chunks".
Chaque chunk représente une portion de texte qui peut être traitée et récupérée 
indépendamment. Les LLMs ont une limite de tokens (ex: 4096, 8192), donc les longs 
documents ne peuvent pas être traités en une seule fois.

Les embeddings sont des représentations numériques (vecteurs) qui capturent le sens 
sémantique des mots et des phrases. Ils permettent de comparer des textes en mesurant 
leur similarité dans un espace vectoriel.

Une base de données vectorielle (comme FAISS, ChromaDB ou Pinecone) stocke et indexe 
les embeddings pour permettre une recherche rapide d'informations pertinentes.
"""

# Créer un objet Document LangChain
document = Document(
    page_content=texte_exemple,
    metadata={"source": "workshop_gomycode", "auteur": "GoMyCode"}
)

print("✅ Document chargé!")
print(f"📄 Nombre de caractères: {len(texte_exemple)}")
print(f"🏷️  Métadonnées: {document.metadata}")

In [ ]:
# Optionnel: Charger un vrai fichier PDF
# Décommentez et adaptez ce code si vous avez un fichier PDF

# from langchain_community.document_loaders import PyPDFLoader
# 
# # Charger un PDF
# loader = PyPDFLoader("mon_document.pdf")
# documents = loader.load()
# 
# print(f"✅ PDF chargé: {len(documents)} pages")
# for i, doc in enumerate(documents[:3]):
#     print(f"\nPage {i+1} (extrait): {doc.page_content[:200]}...")

print("💡 Pour charger un PDF, décommentez le code ci-dessus et fournissez le chemin vers votre fichier.")

## 3.2 Étape 2 — Chunking (Découpage du Document)

> **Pourquoi découper ?**
> - Les LLMs ont une **limite de tokens** (4096, 8192...)
> - Les petits morceaux permettent une **meilleure récupération** d'informations
> - Chaque chunk peut être traité **indépendamment**

```
Document complet (long)
│
├── Chunk 1: "GoMyCode est une startup..."
├── Chunk 2: "Le workshop couvre..."
├── Chunk 3: "Le RAG est une technique..."
└── Chunk 4: "Les embeddings sont..."
```

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Créer le splitter de texte
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # Taille maximale de chaque chunk (en caractères)
    chunk_overlap=50,      # Chevauchement entre chunks (évite de couper le contexte)
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Séparateurs par ordre de priorité
)

# Découper le document en chunks
chunks = text_splitter.split_documents([document])

print(f"✅ Document découpé en {len(chunks)} chunks")
print(f"\n📊 Statistiques:")
tailles = [len(c.page_content) for c in chunks]
print(f"   - Taille moyenne: {sum(tailles)//len(tailles)} caractères")
print(f"   - Plus petit chunk: {min(tailles)} caractères")
print(f"   - Plus grand chunk: {max(tailles)} caractères")

In [ ]:
# Visualiser les chunks
print("📄 Aperçu des chunks créés:")
print("=" * 60)
for i, chunk in enumerate(chunks):
    print(f"\n🔹 Chunk #{i+1} ({len(chunk.page_content)} caractères):")
    print("-" * 40)
    print(chunk.page_content)
    print()

## 3.3 Étape 3 — Embeddings (Représentation Vectorielle)

> **Définition :** Un **embedding** est une représentation numérique (vecteur) qui capture le **sens sémantique** d'un texte.

```
"Le chat mange"    →  [0.23, -0.41, 0.87, ..., 0.12]  (vecteur de 1536 dimensions)
"Le félin dévore"  →  [0.25, -0.38, 0.91, ..., 0.11]  (vecteur similaire!)
"La voiture roule" →  [-0.5, 0.12, -0.23, ..., 0.67]  (vecteur très différent)
```

Les textes **similaires sémantiquement** ont des vecteurs **proches** dans l'espace vectoriel.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# Initialiser le modèle d'embeddings NVIDIA
embeddings_model = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5"  # Modèle d'embedding NVIDIA optimisé pour Q&A
)

# Tester l'embedding sur un texte
texte_test = "Qu'est-ce que le RAG?"
vecteur = embeddings_model.embed_query(texte_test)

print(f"✅ Embedding créé!")
print(f"📊 Texte: '{texte_test}'")
print(f"🔢 Dimension du vecteur: {len(vecteur)}")
print(f"🔢 Premiers 5 éléments: {[round(v, 4) for v in vecteur[:5]]}")

In [ ]:
import numpy as np

def cosine_similarity(v1, v2):
    """Calcule la similarité cosinus entre deux vecteurs."""
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Comparer la similarité entre des phrases
phrases = [
    "Le RAG améliore les LLMs",
    "Le Retrieval Augmented Generation enrichit les modèles de langage",  # Similaire!
    "J'aime manger des pizzas"  # Différent!
]

print("📊 Comparaison de similarité sémantique:")
print("=" * 55)

vecteur_reference = embeddings_model.embed_query(phrases[0])
for phrase in phrases:
    vecteur = embeddings_model.embed_query(phrase)
    similarite = cosine_similarity(vecteur_reference, vecteur)
    emoji = "🟢" if similarite > 0.8 else "🟡" if similarite > 0.5 else "🔴"
    print(f"{emoji} Similarité: {similarite:.4f} | '{phrase}'")

## 3.4 Étape 4 — Base de Données Vectorielle (FAISS)

> **Définition :** FAISS (Facebook AI Similarity Search) est une **base de données vectorielle** qui stocke les embeddings et permet une **recherche ultra-rapide** par similarité.

```
                    🗄️ BASE DE DONNÉES FAISS
                    ┌────────────────────────────────┐
  Chunk 1 ─────►   │ [0.23, -0.41, 0.87, ...] + texte│
  Chunk 2 ─────►   │ [0.15, 0.72, -0.33, ...] + texte│
  Chunk 3 ─────►   │ [-0.5, 0.12, 0.91, ...] + texte │
  ...               │ ...                             │
                    └────────────────────────────────┘
                              │
  Question ──► Embedding ──► 🔍 Recherche des plus proches
                              │
                              ▼
                    📝 Chunks les plus pertinents
```

In [ ]:
from langchain_community.vectorstores import FAISS

# Créer la base de données vectorielle à partir des chunks
print("⏳ Création de la base de données vectorielle...")
print(f"   (Indexation de {len(chunks)} chunks)")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

print("\n✅ Base de données vectorielle FAISS créée!")
print(f"📦 Nombre de vecteurs indexés: {vectorstore.index.ntotal}")

In [ ]:
# Sauvegarder la base de données (optionnel)
vectorstore.save_local("faiss_index")
print("💾 Base de données sauvegardée dans 'faiss_index/'")

# Pour recharger plus tard:
# vectorstore = FAISS.load_local("faiss_index", embeddings_model)

In [ ]:
# Test de recherche dans la base de données
question_test = "Qu'est-ce que le RAG?"

print(f"🔍 Question: '{question_test}'")
print("\n📄 Chunks les plus pertinents trouvés:")
print("=" * 60)

# Recherche des 3 chunks les plus similaires
resultats = vectorstore.similarity_search_with_score(question_test, k=3)

for i, (doc, score) in enumerate(resultats):
    print(f"\n🔹 Résultat #{i+1} (score de distance: {score:.4f})")
    print("-" * 40)
    print(doc.page_content)

## 3.5 Étape 5 — Prompt Template

> **Définition :** Le **Prompt Template** est un modèle structuré qui intègre le **contexte récupéré** dans la question envoyée au LLM.

```
┌─────────────────────────────────────────────────┐
│              PROMPT TEMPLATE                     │
│                                                  │
│  Contexte: {contexte_récupéré_du_vectorstore}   │
│                                                  │
│  Question: {question_de_l_utilisateur}           │
│                                                  │
│  Réponds uniquement en te basant sur le contexte.│
└─────────────────────────────────────────────────┘
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Définir le template de prompt pour le RAG
RAG_TEMPLATE = """
Tu es un assistant IA utile et pédagogique. Réponds à la question de l'utilisateur 
en utilisant UNIQUEMENT les informations fournies dans le contexte ci-dessous.

Si la réponse ne se trouve pas dans le contexte, dis honnêtement: 
"Je ne trouve pas cette information dans les documents fournis."

=== CONTEXTE ===
{context}
================

Question: {question}

Réponse:"""

prompt_template = ChatPromptTemplate.from_template(RAG_TEMPLATE)

# Afficher un exemple du prompt rempli
exemple_contexte = "Le RAG améliore les LLMs en leur donnant accès à des sources externes."
exemple_question = "Comment fonctionne le RAG?"

prompt_rempli = prompt_template.format(context=exemple_contexte, question=exemple_question)
print("📝 Exemple de prompt rempli:")
print("=" * 50)
print(prompt_rempli)

## 3.6 Étape 6 — Assemblage du Pipeline RAG Complet

Maintenant, assemblons toutes les pièces pour créer notre **pipeline RAG complet** !

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Créer le retriever (outil de récupération)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Récupérer les 3 chunks les plus pertinents
)

# Fonction pour formatter les documents récupérés
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Assembler le pipeline RAG avec LCEL (LangChain Expression Language)
rag_chain = (
    {
        "context": retriever | format_docs,   # 1. Récupérer et formater le contexte
        "question": RunnablePassthrough()      # 2. Passer la question telle quelle
    }
    | prompt_template                          # 3. Remplir le template
    | llm                                      # 4. Envoyer au LLM
    | StrOutputParser()                        # 5. Extraire le texte de la réponse
)

print("✅ Pipeline RAG assemblé!")
print("\n🔄 Architecture du pipeline:")
print("   Question → Retriever → Format → Prompt Template → LLM → Réponse")

In [ ]:
# Test du pipeline RAG
def poser_question_rag(question: str):
    """Pose une question au système RAG et affiche la réponse avec le contexte."""
    print(f"👤 Question: {question}")
    print()
    
    # Récupérer le contexte pour information
    docs_contexte = retriever.invoke(question)
    print("📄 Contexte récupéré (chunks pertinents):")
    for i, doc in enumerate(docs_contexte):
        print(f"   [{i+1}] {doc.page_content[:100]}...")
    print()
    
    # Obtenir la réponse du RAG
    reponse = rag_chain.invoke(question)
    print("🤖 Réponse du RAG:")
    print("-" * 50)
    print(reponse)
    print()
    return reponse

# Test 1: Question dans les documents
poser_question_rag("Qu'est-ce que le RAG?")

In [ ]:
# Test 2: Question sur GoMyCode
poser_question_rag("Dans quelles villes est présent GoMyCode?")

In [ ]:
# Test 3: Question hors des documents (pour voir la gestion des inconnues)
poser_question_rag("Quel est le prix de l'abonnement GoMyCode?")

## 3.7 Chatbot RAG Interactif

Construisons maintenant un **chatbot RAG complet avec mémoire** !

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Template pour reformuler la question avec l'historique
CONTEXTUALIZE_TEMPLATE = """Étant donné l'historique de conversation et la dernière question 
de l'utilisateur, reformule la question pour qu'elle soit compréhensible sans l'historique.
NE réponds PAS à la question. Reformule-la seulement si nécessaire, sinon retourne-la telle quelle."""

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", CONTEXTUALIZE_TEMPLATE),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Retriever qui tient compte de l'historique
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

# Template pour la réponse finale avec contexte
QA_TEMPLATE = """Tu es un assistant IA utile et pédagogique. 
Utilise les éléments de contexte suivants pour répondre à la question.
Si tu ne connais pas la réponse, dis que tu ne la trouves pas dans les documents.
Réponds en français.

{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", QA_TEMPLATE),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Créer la chaîne de réponse
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Assembler le RAG chain complet avec mémoire
rag_chain_with_memory = create_retrieval_chain(
    history_aware_retriever, question_answer_chain
)

print("✅ Chatbot RAG avec mémoire créé!")

In [ ]:
class RAGChatbot:
    """
    Chatbot RAG avec mémoire de conversation.
    Combine la recherche dans les documents avec la génération de réponses.
    """
    
    def __init__(self):
        self.chat_history = []
    
    def chat(self, question: str) -> str:
        """Répond à une question en utilisant le RAG."""
        result = rag_chain_with_memory.invoke({
            "input": question,
            "chat_history": self.chat_history
        })
        
        # Mettre à jour l'historique
        self.chat_history.extend([
            HumanMessage(content=question),
            AIMessage(content=result["answer"])
        ])
        
        return result["answer"]
    
    def reset(self):
        """Réinitialise la conversation."""
        self.chat_history = []
        print("🔄 Conversation réinitialisée.")

# Créer le chatbot RAG
rag_bot = RAGChatbot()
print("✅ RAG Chatbot prêt!")

In [ ]:
# Démonstration de la conversation RAG avec mémoire
questions = [
    "Qu'est-ce que GoMyCode?",
    "Combien de villes est-il présent?",  # Question qui nécessite le contexte précédent
    "Explique-moi le concept de chunking.",
    "Pourquoi c'est important?",  # Question de suivi
]

print("💬 DÉMONSTRATION DU CHATBOT RAG")
print("=" * 60)

for q in questions:
    print(f"\n👤 Vous: {q}")
    reponse = rag_bot.chat(q)
    print(f"🤖 Bot RAG: {reponse}")
    print("-" * 40)

## 3.8 Interface Graphique avec Gradio (Bonus)

Créons une interface web simple pour notre chatbot RAG !

In [ ]:
import gradio as gr

# Réinitialiser le bot pour l'interface
rag_bot.reset()

def respond(message, history):
    """Fonction de réponse pour l'interface Gradio."""
    response = rag_bot.chat(message)
    return response

# Créer l'interface Gradio
demo = gr.ChatInterface(
    fn=respond,
    title="🤖 Assistant IA avec RAG — GoMyCode Workshop",
    description="""Posez des questions sur le workshop GoMyCode 'Build your AI Assistant with LLMs & RAG'.
    L'assistant utilise le RAG pour trouver des informations précises dans les documents.""",
    examples=[
        "Qu'est-ce que le RAG?",
        "Comment fonctionne le chunking?",
        "Qu'est-ce que GoMyCode?",
        "Explique les embeddings"
    ],
    theme=gr.themes.Soft()
)

# Lancer l'interface
demo.launch(share=True)  # share=True crée un lien public temporaire

---
## 🎯 Récapitulatif — Ce que nous avons construit

```
┌─────────────────────────────────────────────────────────────────┐
│                    PIPELINE RAG COMPLET                         │
│                                                                 │
│  📄 Documents                                                   │
│      │                                                          │
│      ▼                                                          │
│  ✂️  Chunking  →  Découper en petits morceaux (chunk_size=300) │
│      │                                                          │
│      ▼                                                          │
│  🔢 Embeddings  →  Convertir en vecteurs numériques             │
│      │                                                          │
│      ▼                                                          │
│  🗄️  FAISS  →  Stocker et indexer les vecteurs                  │
│                                                                 │
│  ──────────────────────────────────────────────────────         │
│                                                                 │
│  👤 Question  →  🔢 Embedding  →  🔍 Recherche similarité       │
│                                        │                        │
│                                        ▼                        │
│                               📝 Chunks pertinents              │
│                                        │                        │
│                                        ▼                        │
│                    📋 Prompt Template (Question + Contexte)     │
│                                        │                        │
│                                        ▼                        │
│                               🤖 LLM (GPT-3.5-turbo)           │
│                                        │                        │
│                                        ▼                        │
│                               ✅ Réponse précise!               │
└─────────────────────────────────────────────────────────────────┘
```

### ✅ Compétences acquises :
1. **Concepts fondamentaux** — IA, LLMs, Prompts, Embeddings
2. **Chatbot basique** — Utilisation de l'API OpenAI avec LangChain
3. **Chunking** — Découpage intelligent des documents
4. **Embeddings & Vector Store** — Indexation sémantique avec FAISS
5. **Pipeline RAG** — Architecture complète avec récupération de contexte
6. **Interface utilisateur** — Déploiement avec Gradio

### 🚀 Pour aller plus loin :
- Utiliser **ChromaDB** ou **Pinecone** pour une base de données persistante en production
- Essayer d'autres modèles open source : **Llama 3**, **Mistral**, **Phi-3**
- Ajouter le support de **plusieurs formats de documents** (PDF, Word, CSV, URL)
- Implémenter un **système de feedback** pour améliorer la précision
- Explorer **LangGraph** pour des agents plus sophistiqués

---
## 🧪 Exercice Final — Construisez votre propre RAG !

Maintenant c'est votre tour ! Suivez ces étapes pour créer votre propre assistant RAG sur un document de votre choix.

In [ ]:
# ================================================================
# 🧪 EXERCICE FINAL — Créez votre propre assistant RAG
# ================================================================
# Instructions:
# 1. Remplacez 'votre_texte' par votre propre texte/document
# 2. Ajustez les paramètres de chunking si nécessaire
# 3. Posez des questions sur votre document

# TODO: Remplacez ce texte par votre propre contenu
votre_texte = """
Écrivez ici votre propre texte ou document.
Par exemple:
- Un article de blog
- La description d'un produit
- Un cours de votre domaine
- Des informations sur votre entreprise
"""

# Étape 1: Créer le document
mon_document = Document(
    page_content=votre_texte,
    metadata={"source": "mon_document"}
)

# Étape 2: Chunking
mes_chunks = text_splitter.split_documents([mon_document])
print(f"✅ Document découpé en {len(mes_chunks)} chunks")

# Étape 3: Créer la base vectorielle
ma_vectorstore = FAISS.from_documents(mes_chunks, embeddings_model)
print(f"✅ Base vectorielle créée avec {ma_vectorstore.index.ntotal} vecteurs")

# Étape 4: Créer le retriever et le pipeline RAG
mon_retriever = ma_vectorstore.as_retriever(search_kwargs={"k": 3})

mon_rag_chain = (
    {"context": mon_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("\n✅ Votre assistant RAG est prêt!")
print("💡 Posez vos questions dans la cellule suivante.")

In [ ]:
# TODO: Posez vos propres questions !
ma_question = "Posez votre question ici"  # ← Modifiez cette ligne

reponse = mon_rag_chain.invoke(ma_question)
print(f"👤 Question: {ma_question}")
print(f"\n🤖 Réponse: {reponse}")

---
## 📚 Ressources pour continuer

| Ressource | Lien | Description |
|-----------|------|-------------|
| NVIDIA Build | https://build.nvidia.com | Clé API gratuite + catalogue de modèles |
| NVIDIA NIM | https://docs.api.nvidia.com | Documentation des API NVIDIA |
| LangChain NVIDIA | https://python.langchain.com/docs/integrations/providers/nvidia/ | Intégration LangChain |
| LangChain Docs | https://python.langchain.com | Documentation officielle LangChain |
| FAISS | https://faiss.ai | Documentation FAISS |
| Llama 3.1 | https://huggingface.co/meta-llama | Modèle open source Meta |
| GoMyCode | https://gomycode.com | Continuez à apprendre ! |

---

**🎉 Félicitations ! Vous avez complété le workshop "Build your AI Assistant with LLMs & RAG" !**

*GoMyCode — Empowering the next generation of developers* 🚀